# Notebook 10 – Transfer Learning Implementation



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Experiment plan
Compare scratch training, frozen feature extraction, and partial fine-tuning on the same target split. Record time, data requirement, performance, compute cost, and generalization. Select checkpoints on validation and test once. An ImageNet model needs RGB inputs and its prescribed resize/normalization; never feed raw 32×32 CIFAR images unchanged.

In [2]:
from torchvision.models import resnet18,ResNet18_Weights
USE_PRETRAINED=False # set True to download/cache ImageNet weights
weights=ResNet18_Weights.DEFAULT if USE_PRETRAINED else None
model=resnet18(weights=weights); preprocess=weights.transforms() if weights else None
print('preprocess:',preprocess,'original head:',model.fc)
for p in model.parameters(): p.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,10); model=model.to(device)
optimizer=torch.optim.Adam(model.fc.parameters(),lr=1e-3)
print('classifier-only trainable:',sum(p.numel() for p in model.parameters() if p.requires_grad))
for p in model.layer4.parameters(): p.requires_grad=True
optimizer=torch.optim.Adam([{'params':model.fc.parameters(),'lr':1e-3},{'params':model.layer4.parameters(),'lr':1e-4}])
print('partial fine-tuning trainable:',sum(p.numel() for p in model.parameters() if p.requires_grad))
# For each stage, train on train_loader and evaluate on the same deterministic val_loader.

preprocess: None original head: Linear(in_features=512, out_features=1000, bias=True)
classifier-only trainable: 5130
partial fine-tuning trainable: 8398858


## Results and interpretation
| Method | Training time | Data requirement | Validation/test score | Compute cost | Generalization notes |
|---|---:|---:|---:|---:|---|
| Scratch | measure | often higher | measure | full model | learns target-specific features |
| Frozen backbone | measure | lower | measure | classifier only | may underfit domain mismatch |
| Partial fine-tune | measure | moderate | measure | selected layers | adapts features; overfit risk |

Use the same metric/split for all methods and report hardware, image size, batch size, epochs, and weight source/cache status. Explain differences in training time, data need, performance, compute, and generalization.

## End-to-end optional CIFAR-10 comparison
Set `RUN_TRANSFER_EXPERIMENT=True` to download CIFAR-10 and ImageNet weights, then compare frozen-backbone training, partial fine-tuning, and a randomly initialized ResNet-18 on matched train/validation/test subsets. This compute-heavy experiment is opt-in; a GPU is recommended.


In [3]:
RUN_TRANSFER_EXPERIMENT=False
if RUN_TRANSFER_EXPERIMENT:
    from torchvision import datasets,transforms
    from torch.utils.data import DataLoader,Subset
    from torchvision.models import resnet18,ResNet18_Weights
    import copy
    weights=ResNet18_Weights.DEFAULT
    mean,std=weights.transforms().mean,weights.transforms().std
    train_tf=transforms.Compose([transforms.Resize(256),transforms.RandomCrop(224),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean,std)])
    eval_tf=weights.transforms()
    train_aug=datasets.CIFAR10('data',train=True,download=True,transform=train_tf)
    train_eval=datasets.CIFAR10('data',train=True,download=False,transform=eval_tf)
    test_set=datasets.CIFAR10('data',train=False,download=True,transform=eval_tf)
    generator=torch.Generator().manual_seed(42); selected=torch.randperm(len(train_aug),generator=generator)[:512].tolist()
    train_ids,valid_ids=selected[:400],selected[400:]
    train_loader=DataLoader(Subset(train_aug,train_ids),batch_size=16,shuffle=True,num_workers=0)
    val_loader=DataLoader(Subset(train_eval,valid_ids),batch_size=32,shuffle=False,num_workers=0)
    test_loader=DataLoader(Subset(test_set,range(128)),batch_size=32,shuffle=False,num_workers=0)
    def new_resnet(pretrained):
        selected_weights=ResNet18_Weights.DEFAULT if pretrained else None
        model=resnet18(weights=selected_weights); model.fc=nn.Linear(model.fc.in_features,10)
        return model.to(device)
    def score(model,loader):
        model.eval(); good=total=0
        with torch.no_grad():
            for xb,yb in loader:
                pred=model(xb.to(device)).argmax(1).cpu(); good+=(pred==yb).sum().item(); total+=len(yb)
        return good/total
    def train_stage(model,optimizer,epochs=1,freeze_bn=False):
        start=time.perf_counter(); best=0.; best_state=copy.deepcopy(model.state_dict())
        for _ in range(epochs):
            model.train()
            if freeze_bn:
                for module in model.modules():
                    if isinstance(module,nn.BatchNorm2d): module.eval()
            for xb,yb in train_loader:
                xb,yb=xb.to(device),yb.to(device); optimizer.zero_grad(); loss=nn.functional.cross_entropy(model(xb),yb); loss.backward(); optimizer.step()
            val_acc=score(model,val_loader)
            if val_acc>best: best=val_acc; best_state=copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        return best,time.perf_counter()-start
    transfer=new_resnet(True)
    for parameter in transfer.parameters(): parameter.requires_grad=False
    for parameter in transfer.fc.parameters(): parameter.requires_grad=True
    head_acc,head_time=train_stage(transfer,torch.optim.Adam(transfer.fc.parameters(),lr=1e-3),epochs=1,freeze_bn=True)
    head_state=copy.deepcopy(transfer.state_dict())
    for parameter in transfer.layer4.parameters(): parameter.requires_grad=True
    tune_optimizer=torch.optim.Adam([{'params':transfer.fc.parameters(),'lr':1e-3},{'params':transfer.layer4.parameters(),'lr':1e-4}])
    tune_acc,tune_time=train_stage(transfer,tune_optimizer,epochs=1,freeze_bn=True)
    tune_state=copy.deepcopy(transfer.state_dict())
    scratch=new_resnet(False)
    for parameter in scratch.parameters(): parameter.requires_grad=True
    scratch_acc,scratch_time=train_stage(scratch,torch.optim.Adam(scratch.parameters(),lr=1e-3),epochs=1)
    print('method | best validation accuracy | train seconds | test accuracy')
    for name,model,val_acc,seconds,state in [('frozen backbone',transfer,head_acc,head_time,head_state),('partial fine-tune',transfer,tune_acc,tune_time,tune_state),('scratch',scratch,scratch_acc,scratch_time,scratch.state_dict())]:
        model.load_state_dict(state)
        print(name,round(val_acc,3),round(seconds,1),round(score(model,test_loader),3))
else:
    print('Set RUN_TRANSFER_EXPERIMENT=True to run the CIFAR-10 comparison; pretrained weights and dataset will download.')


Set RUN_TRANSFER_EXPERIMENT=True to run the CIFAR-10 comparison; pretrained weights and dataset will download.
